# 🩺 Diabetic Retinopathy Grading — Production Pipeline v19
## All 30 Steps | QWK ≥ 0.90 Target | Colab · Kaggle · Windows · Linux

---
| # | Step |
|---|------|
| 1 | Setup — GPU / CPU / Storage check |
| 2 | Install Requirements |
| 3 | Kaggle Upload & Authentication |
| 4 | Dataset Download & Extraction |
| 5 | Load Dataset |
| 6 | Data Cleaning (Laplacian + intensity) |
| 7 | EDA |
| 8 | Label Analysis |
| 9 | Preprocessing — Strict Pipeline |
| 10 | Preprocessing Cache |
| 11 | Train / Test Split |
| 12 | K-Fold (StratifiedKFold = 5) |
| 13 | Augmentation + Dataset Pipeline |
| 14 | DataLoader Creation |
| 15 | Model Initialization |
| 16 | Loss + Optimizer + Scheduler |
| 17 | Checkpoint & Resume System |
| 18 | Training State Management |
| 19 | Cross-Validation Training (5-Fold Execution) |
| 20 | Training — Phase-Wise Strategy |
| 21 | Validation (per epoch, per fold) |
| 22 | Early Stopping |
| 23 | OOF Predictions |
| 24 | TTA — Test-Time Augmentation |
| 25 | Threshold Optimization |
| 26 | Testing — Final Hold-Out |
| 27 | Metrics & Evaluation |
| 28 | Model Export |
| 29 | Grad-CAM++ Explainability |
| 30 | Deployment — Streamlit + Hugging Face |


## 🖥️ Step 1 — Setup: GPU / CPU / Storage Check

In [ ]:
import sys, os, shutil, platform, subprocess
from pathlib import Path

print("=" * 65)
print("  SYSTEM DIAGNOSTICS")
print("=" * 65)
print(f"  Python      : {sys.version.split()[0]}")
print(f"  Platform    : {platform.system()} {platform.machine()}")

# ── GPU / device check ────────────────────────────────────────
try:
    import torch
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f"  GPU {i}       : {p.name}")
            print(f"  VRAM        : {p.total_memory/1e9:.1f} GB")
        print(f"  CUDA        : {torch.version.cuda}")
        print(f"  cuDNN       : {torch.backends.cudnn.version()}")
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        print("  Device      : Apple MPS (M-series GPU)")
    else:
        print("  Device      : CPU only  ⚠️  Training will be slow")
    print(f"  PyTorch     : {torch.__version__}")
except ImportError:
    print("  PyTorch     : not installed yet")

# ── Storage check ─────────────────────────────────────────────
total, used, free = shutil.disk_usage(Path.home())
print(f"  Disk Free   : {free/1e9:.1f} GB  (need ≥ 5 GB)")
if free < 5e9:
    print("  ⚠️  WARNING: Low disk space — free up at least 5 GB!")

# ── RAM check ─────────────────────────────────────────────────
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f"  RAM Total   : {vm.total/1e9:.1f} GB")
    print(f"  RAM Free    : {vm.available/1e9:.1f} GB")
except ImportError:
    pass

print("=" * 65)
print("✅ Step 1 complete — System check done.")


## 📦 Step 2 — Install Requirements

In [ ]:
import sys, subprocess

def _pip(*args):
    """Run pip; return (returncode, stderr)."""
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + list(args),
        capture_output=True, text=True)
    return r.returncode, r.stderr

# ── Standard packages ─────────────────────────────────────────
PACKAGES = [
    "timm>=1.0.3",
    "albumentations>=1.4.0,<2.0.0",
    "opencv-python-headless",
    "scikit-learn",
    "pandas",
    "numpy",
    "tqdm",
    "matplotlib",
    "scipy",
    "pyarrow",
    "fastparquet",
    "kaggle",
    "psutil",
    "streamlit",
]

print("Installing / upgrading packages ...")
failed = []
for pkg in PACKAGES:
    print(f"  {pkg:<40s}", end=" ", flush=True)
    rc, err = _pip("--upgrade", pkg)
    if rc == 0:
        print("✅")
    else:
        print("❌"); print(err[-300:]); failed.append(pkg)

# ── packaging (Anaconda ships it without RECORD → force-reinstall) ──
print(f"  {'packaging':<40s}", end=" ", flush=True)
rc, err = _pip("--force-reinstall", "--no-deps", "packaging")
print("✅" if rc == 0 else f"❌ {err[-200:]}")
if rc != 0: failed.append("packaging")

# ── pytorch-grad-cam (PyPI → GitHub fallback) ──────────────────
print(f"  {'pytorch-grad-cam':<40s}", end=" ", flush=True)
rc, _ = _pip("--upgrade", "pytorch-grad-cam")
if rc != 0:
    print("(trying GitHub) ...", end=" ", flush=True)
    rc2, err2 = _pip("git+https://github.com/jacobgil/pytorch-grad-cam.git")
    print("✅ (GitHub)" if rc2 == 0 else "⚠️  skipped — Step 29 will be graceful")
else:
    print("✅")

if failed:
    raise RuntimeError(f"Critical packages failed: {failed}")
print("\n✅ Step 2 complete — All packages installed.")


## 🔑 Step 3 — Kaggle File Upload & Authentication

In [ ]:
import os, sys, json, warnings, zipfile, gc, time, random, shutil
from pathlib import Path
from copy import deepcopy
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from tqdm.auto import tqdm
from packaging.version import Version

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    cohen_kappa_score, accuracy_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
)
from scipy.optimize import minimize

warnings.filterwarnings("ignore")

# ── Global Config ─────────────────────────────────────────────
CFG = {
    "seed"            : 42,
    "model_name"      : "tf_efficientnetv2_b1",
    "n_folds"         : 5,
    "test_size"       : 0.10,
    "lr"              : 3e-4,
    "min_lr"          : 1e-6,
    "weight_decay"    : 1e-4,
    "patience"        : 5,
    "min_delta"       : 0.001,
    "grad_clip"       : 1.0,
    "label_smooth"    : 0.05,
    "dropout"         : 0.50,
    "blur_threshold"  : 50.0,
    "dark_threshold"  : 15.0,
    "nonblack_ratio"  : 0.10,
    "phases": [
        {"id":1, "size":224, "batch_size":32, "epochs":15, "freeze":True},
        {"id":2, "size":384, "batch_size":16, "epochs":40, "freeze":False},
        {"id":3, "size":512, "batch_size": 8, "epochs":25, "freeze":False},
    ],
}

def seed_everything(seed=CFG["seed"]):
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False
seed_everything()

# ── Device ────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"🔥 GPU : {torch.cuda.get_device_name(0)}  "
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB VRAM")
elif hasattr(torch.backends,"mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps"); print("🍎 Apple MPS")
else:
    DEVICE = torch.device("cpu");  print("💻 CPU only")
USE_AMP = (DEVICE.type == "cuda")

# ── Environment ───────────────────────────────────────────────
IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/input")

# ── Paths ─────────────────────────────────────────────────────
if IN_KAGGLE:
    DATA_DIR     = Path("/kaggle/input/aptos2019-blindness-detection")
    ARTIFACT_DIR = Path("/kaggle/working/dr_v19")
elif IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception: pass
    DATA_DIR     = Path("/content/drive/MyDrive/DR_data/aptos2019")
    ARTIFACT_DIR = Path("/content/drive/MyDrive/DR_data/artifacts_v19")
else:
    DATA_DIR     = Path(os.environ.get("DR_DATA",     str(Path.home()/"DR_data"/"aptos2019")))
    ARTIFACT_DIR = Path(os.environ.get("DR_ARTIFACTS",str(Path.home()/"DR_data"/"artifacts_v19")))

IMG_DIR    = DATA_DIR / "train_images"
CSV_PATH   = DATA_DIR / "train.csv"
CACHE_DIR  = ARTIFACT_DIR / "cache"
PLOT_DIR   = ARTIFACT_DIR / "plots"
EXPORT_DIR = ARTIFACT_DIR / "export"
for d in [ARTIFACT_DIR, CACHE_DIR, PLOT_DIR, EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

NUM_CLASSES   = 5
GRADE_MAP     = {0:"No DR",1:"Mild",2:"Moderate",3:"Severe",4:"Proliferative"}
GRADE_COLORS  = ["#2ecc71","#f1c40f","#e67e22","#e74c3c","#8e44ad"]
IMAGENET_MEAN = [0.485,0.456,0.406]
IMAGENET_STD  = [0.229,0.224,0.225]
CACHE_SIZE    = 224

# ── State helpers ─────────────────────────────────────────────
_STATE = ARTIFACT_DIR / "state.json"
def st_load(): return json.loads(_STATE.read_text()) if _STATE.exists() else {}
def st_save(k,v): s=st_load(); s[k]=v; _STATE.write_text(json.dumps(s,indent=2))

def safe_load(path, map_location="cpu"):
    try:    return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError: return torch.load(path, map_location=map_location)

# ── Kaggle Auth ───────────────────────────────────────────────
if IN_KAGGLE:
    print("✅ Kaggle Notebook — auth handled automatically.")
else:
    kaggle_cfg = Path.home() / ".kaggle" / "kaggle.json"
    env_ok = bool(os.environ.get("KAGGLE_KEY") and os.environ.get("KAGGLE_USERNAME"))

    if not kaggle_cfg.exists() and not env_ok:
        if IN_COLAB:
            from google.colab import files as _cf
            print("📤 Upload your kaggle.json now ...")
            _up = _cf.upload()
            kaggle_cfg.parent.mkdir(parents=True, exist_ok=True)
            for _fn, _data in _up.items():
                kaggle_cfg.write_bytes(_data)
            print(f"   Saved to {kaggle_cfg}")
        else:
            print("=" * 60)
            print("  ⚙️  KAGGLE SETUP (one-time only)")
            print("  1. https://www.kaggle.com/settings/account")
            print("  2. API → Create New Token → kaggle.json downloaded")
            print(f"  3. Move it to: {kaggle_cfg.parent}")
            print("  OR set env vars: KAGGLE_USERNAME + KAGGLE_KEY")
            print("=" * 60)

    if kaggle_cfg.exists():
        try: kaggle_cfg.chmod(0o600)
        except Exception: pass
        print(f"✅ kaggle.json found at {kaggle_cfg}")
    elif env_ok:
        print("✅ Kaggle env vars set.")
    else:
        print("⚠️  No Kaggle credentials — download will fail in Step 4.")

print(f"\n✅ Step 3 complete | PyTorch {torch.__version__} | timm {timm.__version__}")
print(f"   DATA_DIR     : {DATA_DIR}")
print(f"   ARTIFACT_DIR : {ARTIFACT_DIR}")


## 📥 Step 4 — Dataset Download & Extraction (APTOS 2019)

In [ ]:
COMPETITION = "aptos2019-blindness-detection"

def _dataset_ready():
    return CSV_PATH.exists() and len(list(IMG_DIR.glob("*.png"))) >= 3000

if _dataset_ready():
    print(f"✅ Dataset already present: {len(list(IMG_DIR.glob('*.png'))):,} images.")

elif IN_KAGGLE:
    src = Path(f"/kaggle/input/{COMPETITION}")
    if src.exists():
        if not DATA_DIR.exists():
            DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
            DATA_DIR.symlink_to(src)
        print(f"✅ Kaggle: dataset linked {src} → {DATA_DIR}")
    else:
        print(f"⚠️  Add '{COMPETITION}' as a dataset in the Kaggle right panel.")

else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    _zip = DATA_DIR / f"{COMPETITION}.zip"

    if not _zip.exists():
        print("⬇️  Downloading APTOS 2019 (~1.5 GB) ...")
        r = subprocess.run(
            [sys.executable, "-m", "kaggle", "competitions", "download",
             "-c", COMPETITION, "-p", str(DATA_DIR)],
            capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f"Download failed:\n{r.stderr}")
        print("✅ Download complete.")
    else:
        print(f"✅ ZIP exists: {_zip}")

    print("📦 Extracting archive ...")
    with zipfile.ZipFile(_zip, "r") as z:
        z.extractall(DATA_DIR)
    for nz in DATA_DIR.glob("*.zip"):
        with zipfile.ZipFile(nz, "r") as z: z.extractall(DATA_DIR)
        nz.unlink()
    _zip.unlink(missing_ok=True)

# ── Verification ──────────────────────────────────────────────
print()
for label, path in [("train.csv", CSV_PATH), ("train_images/", IMG_DIR)]:
    ok = path.exists()
    extra = f" ({len(list(path.glob('*.png')))} images)" if ok and path.is_dir() else ""
    print(f"  {'✅' if ok else '❌'} {label}{extra}")

if not _dataset_ready():
    raise RuntimeError("Dataset incomplete — check errors above.")
print("\n✅ Step 4 complete — Dataset ready.")


## 📂 Step 5 — Load Dataset (train.csv + image paths + label mapping)

In [ ]:
df_raw = pd.read_csv(CSV_PATH)
df_raw["path"]  = df_raw["id_code"].apply(lambda x: str(IMG_DIR / f"{x}.png"))
df_raw["label"] = df_raw["diagnosis"].map(GRADE_MAP)

# Drop missing image files
exists_mask = df_raw["path"].apply(lambda p: Path(p).exists())
n_missing   = (~exists_mask).sum()
if n_missing:
    print(f"⚠️  {n_missing} rows dropped — image files missing.")
df_raw = df_raw[exists_mask].reset_index(drop=True)

print(f"✅ {len(df_raw):,} images loaded.")
print(f"\nLabel mapping:")
for g in range(NUM_CLASSES):
    n = (df_raw.diagnosis==g).sum()
    print(f"  {g} → {GRADE_MAP[g]:<15s}  {n:5d} samples")

print("\n✅ Step 5 complete.")


## 🧹 Step 6 — Data Cleaning (Laplacian blur + intensity + black-border checks)

In [ ]:
def check_image_quality(path):
    """
    Returns (is_ok: bool, reason: str).
    Checks:
      - Unreadable file
      - Blurry (Laplacian variance < blur_threshold)
      - Too dark (mean brightness < dark_threshold)
      - Mostly black borders (non-black ratio < nonblack_ratio)
    """
    bgr = cv2.imread(str(path))
    if bgr is None:
        return False, "unreadable"

    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    if lap_var < CFG["blur_threshold"]:
        return False, f"blurry(lap={lap_var:.1f})"

    mean_br = float(gray.mean())
    if mean_br < CFG["dark_threshold"]:
        return False, f"too_dark(br={mean_br:.1f})"

    nonblack = float((gray > 10).mean())
    if nonblack < CFG["nonblack_ratio"]:
        return False, f"black_border(nb={nonblack:.2f})"

    return True, ""

_clean_cache = ARTIFACT_DIR / "clean_flags.parquet"
if _clean_cache.exists():
    df_flags = pd.read_parquet(_clean_cache)
    print("✅ [RESUME] Quality flags loaded from cache.")
else:
    print(f"Running quality check on {len(df_raw)} images ...")
    results = [check_image_quality(p) for p in tqdm(df_raw["path"], leave=False)]
    df_raw["is_ok"]  = [r[0] for r in results]
    df_raw["reason"] = [r[1] for r in results]
    df_flags = df_raw[["id_code","is_ok","reason"]].copy()
    df_flags.to_parquet(_clean_cache, index=False)

if "is_ok" not in df_raw.columns:
    df_raw = df_raw.merge(df_flags, on="id_code", how="left")

bad  = df_raw[~df_raw["is_ok"]]
good = df_raw[ df_raw["is_ok"]].reset_index(drop=True)

print(f"  Total    : {len(df_raw):,}")
print(f"  Removed  : {len(bad):,}  (reason breakdown below)")
if len(bad): print(bad["reason"].value_counts().to_string())
print(f"  Kept     : {len(good):,} clean images")

df = good.copy()
print("\n✅ Step 6 complete.")


## 📊 Step 7 — EDA (Exploratory Data Analysis)

In [ ]:
_eda_done = PLOT_DIR / "_eda_done.flag"
if _eda_done.exists():
    print("✅ [RESUME] EDA already done.")
    for p in PLOT_DIR.glob("eda_*.png"):
        plt.figure(figsize=(10,4))
        plt.imshow(plt.imread(str(p))); plt.axis("off")
        plt.title(p.stem); plt.show()
else:
    counts = [(df.diagnosis==g).sum() for g in range(NUM_CLASSES)]
    xlabels= [f"G{g}\n{GRADE_MAP[g]}" for g in range(NUM_CLASSES)]

    # Plot 1: Distribution
    fig, axes = plt.subplots(1, 2, figsize=(14,5))
    bars = axes[0].bar(xlabels, counts, color=GRADE_COLORS, edgecolor="k", lw=0.6)
    axes[0].set_title("Class Distribution (Count)", fontweight="bold", fontsize=13)
    axes[0].set_ylabel("Number of Images")
    for b,n in zip(bars, counts):
        axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+20, str(n), ha="center", fontsize=9)
    axes[1].pie([c/sum(counts)*100 for c in counts], labels=xlabels,
                colors=GRADE_COLORS, autopct="%1.1f%%", startangle=140, textprops={"fontsize":9})
    axes[1].set_title("Class Distribution (%)", fontweight="bold", fontsize=13)
    plt.tight_layout()
    plt.savefig(str(PLOT_DIR/"eda_distribution.png"), dpi=120, bbox_inches="tight")
    plt.show()

    # Plot 2: Sample grid (5 grades x 4 samples)
    fig2, ax2 = plt.subplots(NUM_CLASSES, 4, figsize=(14, 18))
    for g in range(NUM_CLASSES):
        samples = df[df.diagnosis==g].sample(min(4, counts[g]), random_state=42)
        for j, (_, row) in enumerate(samples.iterrows()):
            bgr = cv2.imread(str(row.path))
            img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB) if bgr is not None \
                  else np.zeros((256,256,3), np.uint8)
            ax2[g][j].imshow(img)
            ax2[g][j].set_title(f"G{g}: {GRADE_MAP[g]}", fontsize=8) if j==0 else None
            ax2[g][j].axis("off")
    fig2.suptitle("Sample Fundus Images per DR Grade", fontsize=14, fontweight="bold")
    fig2.tight_layout()
    fig2.savefig(str(PLOT_DIR/"eda_samples.png"), dpi=100, bbox_inches="tight")
    plt.show(fig2)

    # Plot 3: Brightness distribution
    fig3, ax3 = plt.subplots(1,1, figsize=(10,4))
    for g in range(NUM_CLASSES):
        brightness = []
        for p in df[df.diagnosis==g].path.sample(min(200,counts[g]), random_state=42):
            bgr = cv2.imread(str(p))
            if bgr is not None:
                brightness.append(cv2.cvtColor(bgr,cv2.COLOR_BGR2GRAY).mean())
        ax3.hist(brightness, bins=30, alpha=0.5, color=GRADE_COLORS[g], label=f"G{g}")
    ax3.set_title("Brightness Distribution per Grade", fontweight="bold")
    ax3.set_xlabel("Mean Pixel Brightness"); ax3.legend()
    fig3.tight_layout()
    fig3.savefig(str(PLOT_DIR/"eda_brightness.png"), dpi=120, bbox_inches="tight")
    plt.show(fig3)

    _eda_done.touch()
    print("✅ Step 7 complete — EDA saved.")


## ⚖️ Step 8 — Label Analysis (Class Imbalance → Weights & Sampler)

In [ ]:
counts_arr = np.array([(df.diagnosis==g).sum() for g in range(NUM_CLASSES)], dtype=float)
imbalance  = counts_arr.max() / counts_arr.min()

# Class weights: inversely proportional to frequency, normalized
cls_w_np = len(df) / (NUM_CLASSES * np.maximum(counts_arr, 1))
cls_w_np = cls_w_np / cls_w_np.sum() * NUM_CLASSES   # normalize to sum=NUM_CLASSES
CLASS_WEIGHTS = torch.tensor(cls_w_np, dtype=torch.float32)

print("Label / Imbalance Analysis")
print("-" * 55)
print(f"  Total images   : {len(df):,}")
print(f"  Imbalance ratio: {imbalance:.1f}x  "
      f"({'SEVERE' if imbalance>10 else 'MODERATE' if imbalance>4 else 'MILD'})")
print()
print(f"  {'Grade':<6} {'Name':<15} {'Count':>6} {'%':>6}  {'Weight':>7}  Bar")
print("-" * 55)
for g in range(NUM_CLASSES):
    n   = int(counts_arr[g])
    pct = n / len(df) * 100
    w   = cls_w_np[g]
    bar = "█" * max(1, int(w * 8))
    print(f"  G{g}    {GRADE_MAP[g]:<15} {n:>6} {pct:>5.1f}%  {w:>7.3f}  {bar}")
print("-" * 55)
print(f"\n  CLASS_WEIGHTS = {np.round(cls_w_np,3)}")
print("  → Used in: WeightedCrossEntropyLoss + WeightedRandomSampler")
print("\n✅ Step 8 complete.")


## 🔬 Step 9 — Preprocessing: Strict Pipeline

```
1. Load BGR → RGB
2. Auto-crop black borders  (mask > 7)
3. Pad to square  (aspect ratio preserved)
4. Resize to target size
5. Circular retinal mask
6. CLAHE on LAB L-channel
7. Ben Graham: 4×img − 4×GaussBlur + 128
```


In [ ]:
def preprocess_fundus(path_or_img, size=512, sigma_ratio=10, apply_clahe=True):
    """
    STRICT fundus preprocessing pipeline.
    Returns uint8 RGB array of shape (size, size, 3).
    """
    # 0. Load
    if isinstance(path_or_img, np.ndarray):
        img = path_or_img.copy()
    else:
        bgr = cv2.imread(str(path_or_img))
        if bgr is None:
            return np.zeros((size, size, 3), np.uint8)
        img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # 1. Auto-crop black borders
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > 7
    if mask.any():
        rows = np.where(mask.any(1))[0]
        cols = np.where(mask.any(0))[0]
        img  = img[rows[0]:rows[-1]+1, cols[0]:cols[-1]+1]

    # 2. Pad to square
    h, w = img.shape[:2]
    S    = max(h, w)
    ph   = (S-h)//2;  pb = S-h-ph
    pw   = (S-w)//2;  pr = S-w-pw
    img  = cv2.copyMakeBorder(img, ph, pb, pw, pr, cv2.BORDER_CONSTANT, value=0)

    # 3. Resize
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)

    # 4. Circular retinal mask
    cmask = np.zeros((size, size), np.uint8)
    cv2.circle(cmask, (size//2, size//2), int(size//2 * 0.97), 255, -1)
    img[cmask == 0] = 0

    # 5. CLAHE on LAB L-channel
    if apply_clahe:
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        lab[:,:,0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(lab[:,:,0])
        img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    # 6. Ben Graham sharpening: 4*img - 4*blur + 128
    sigma = max(int(size / sigma_ratio) | 1, 1)
    blur  = cv2.GaussianBlur(img, (0, 0), sigmaX=sigma)
    img   = cv2.addWeighted(img, 4, blur, -4, 128)
    img[cmask == 0] = 0

    return img

# Visual sanity check
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for g in range(4):
    sample_path = df[df.diagnosis==g].path.iloc[0]
    bgr_raw = cv2.imread(str(sample_path))
    raw_rgb = cv2.cvtColor(bgr_raw, cv2.COLOR_BGR2RGB)
    proc    = preprocess_fundus(sample_path, size=512)
    axes[0][g].imshow(raw_rgb); axes[0][g].set_title(f"RAW G{g}", fontsize=8); axes[0][g].axis("off")
    axes[1][g].imshow(proc);    axes[1][g].set_title(f"PROC G{g}", fontsize=8); axes[1][g].axis("off")
plt.suptitle("Raw vs Preprocessed (Ben Graham + CLAHE)", fontweight="bold")
plt.tight_layout()
plt.savefig(str(PLOT_DIR/"preprocessing_comparison.png"), dpi=100, bbox_inches="tight")
plt.show()

# Latency check
t0 = time.time()
for _ in range(5): preprocess_fundus(df.path.iloc[0], size=512)
print(f"\nPreprocess latency : {(time.time()-t0)/5*1e3:.1f} ms/img @ 512px")
print("✅ Step 9 complete.")


## 💽 Step 10 — Preprocessing Cache (save processed images to disk)

In [ ]:
def cached_path(orig_path):
    return CACHE_DIR / f"{Path(orig_path).stem}.png"

_cache_flag = CACHE_DIR / "_done.flag"
if _cache_flag.exists():
    n_cached = len(list(CACHE_DIR.glob("*.png")))
    print(f"✅ [RESUME] Cache exists: {n_cached:,} images at {CACHE_DIR}")
else:
    print(f"Building cache ({len(df):,} images @ {CACHE_SIZE}px) ...")
    n_done = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Caching"):
        dest = cached_path(row.path)
        if not dest.exists():
            proc = preprocess_fundus(row.path, size=CACHE_SIZE)
            cv2.imwrite(str(dest), cv2.cvtColor(proc, cv2.COLOR_RGB2BGR))
        n_done += 1
    _cache_flag.touch()
    print(f"✅ Cache built: {n_done:,} images → {CACHE_DIR}")

# Verify a cached image
_test_cached = cached_path(df.path.iloc[0])
_test_img    = cv2.imread(str(_test_cached))
print(f"   Sample cached image shape: {_test_img.shape}")
print("✅ Step 10 complete.")


## ✂️ Step 11 — Train / Test Split (hold-out, never used in training)

In [ ]:
_split_cache = ARTIFACT_DIR / "train_test_split.parquet"
if _split_cache.exists():
    _sp = pd.read_parquet(_split_cache)
    df["split"] = _sp["split"].values
    print("✅ [RESUME] Split loaded.")
else:
    train_idx, test_idx = train_test_split(
        df.index,
        test_size=CFG["test_size"],
        stratify=df["diagnosis"],
        random_state=CFG["seed"])
    df["split"] = "train"
    df.loc[test_idx, "split"] = "test"
    df[["id_code","split"]].to_parquet(_split_cache, index=False)
    print("✅ Split created.")

df_trainval = df[df["split"]=="train"].reset_index(drop=True)
df_test     = df[df["split"]=="test" ].reset_index(drop=True)

print(f"   Train+Val : {len(df_trainval):,}")
print(f"   Test      : {len(df_test):,}  (HELD-OUT — never used in training)")
print(f"\n   Test grade distribution:")
for g in range(NUM_CLASSES):
    n = (df_test.diagnosis==g).sum()
    print(f"     G{g} {GRADE_MAP[g]:<15s}: {n}")
print("\n✅ Step 11 complete.")


## 🔀 Step 12 — K-Fold: StratifiedKFold = 5 (applied only on train+val)

In [ ]:
_kfold_cache = ARTIFACT_DIR / "kfold_splits.parquet"
if _kfold_cache.exists():
    _kf = pd.read_parquet(_kfold_cache)
    df_trainval["fold"] = _kf["fold"].values
    print("✅ [RESUME] K-Fold splits loaded.")
else:
    skf = StratifiedKFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["seed"])
    df_trainval["fold"] = -1
    for fi, (_, vi) in enumerate(skf.split(df_trainval, df_trainval["diagnosis"])):
        df_trainval.loc[vi, "fold"] = fi
    df_trainval[["id_code","fold"]].to_parquet(_kfold_cache, index=False)
    print("✅ 5-Fold splits created.")

print(f"\n  {CFG['n_folds']}-Fold distribution (class balance preserved):")
for f in range(CFG["n_folds"]):
    n  = (df_trainval.fold==f).sum()
    gd = df_trainval[df_trainval.fold==f].diagnosis.value_counts().sort_index()
    gs = "  ".join(f"G{g}:{c}" for g,c in gd.items())
    print(f"  Fold {f}:  {n:5d} imgs   {gs}")
print("\n✅ Step 12 complete.")


## 🔄 Step 13 — Augmentation + Dataset Pipeline

- RandomResizedCrop (0.8–1.0)
- HorizontalFlip + VerticalFlip
- Rotate ±15°, ShiftScaleRotate
- RandomBrightnessContrast
- CLAHE (p=0.3)
- GaussianNoise (light)
- CoarseDropout (small)
- Custom Dataset class with preprocessing
- WeightedRandomSampler (MANDATORY for imbalance)


In [ ]:
_A_NEW = Version(A.__version__) >= Version("1.4.0")
print(f"Albumentations {A.__version__} — API: {'new ≥1.4' if _A_NEW else 'legacy <1.4'}")

def _gauss_noise():
    return (A.GaussNoise(std_range=(0.03,0.10), p=0.2) if _A_NEW
            else A.GaussNoise(var_limit=(10.0,40.0), p=0.2))

def _coarse_drop(sz):
    h = sz // 16
    return (A.CoarseDropout(num_holes_range=(1,6), hole_height_range=(h,h),
                            hole_width_range=(h,h), p=0.2) if _A_NEW
            else A.CoarseDropout(max_holes=6, max_height=h, max_width=h, p=0.2))

def get_train_transform(sz):
    return A.Compose([
        A.RandomResizedCrop(height=sz, width=sz, scale=(0.8,1.0), ratio=(0.9,1.1)),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
        A.CLAHE(clip_limit=2.0, p=0.3),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
        A.RandomGamma(gamma_limit=(80,120), p=0.3),
        _gauss_noise(),
        A.MotionBlur(blur_limit=3, p=0.1),
        _coarse_drop(sz),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_val_transform(sz):
    return A.Compose([
        A.Resize(sz, sz),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

class DRDataset(Dataset):
    """
    Custom DR Dataset.
    - Loads from cache (fast) when size == CACHE_SIZE.
    - Falls back to on-the-fly preprocessing for other sizes.
    """
    def __init__(self, df, transform=None, img_size=512, use_cache=True):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.img_size  = img_size
        self.use_cache = use_cache

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Load: cache first, else preprocess on-the-fly
        cp = cached_path(row.path)
        if self.use_cache and cp.exists() and self.img_size == CACHE_SIZE:
            bgr = cv2.imread(str(cp))
            img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        else:
            img = preprocess_fundus(row.path, size=self.img_size)

        if self.transform:
            img = self.transform(image=img)["image"]
        else:
            img = torch.from_numpy(img.transpose(2,0,1)).float() / 255.0

        label = torch.tensor(int(row.diagnosis), dtype=torch.long)
        return img, label

print("✅ Step 13 complete — Transforms & DRDataset defined.")


## 🚚 Step 14 — DataLoader Creation (train / val / test — optimized)

In [ ]:
def make_weighted_loader(df_split, dataset, batch_size, drop_last=False):
    """
    DataLoader with WeightedRandomSampler.
    MANDATORY for class imbalance — ensures every batch reflects the desired class dist.
    """
    labels  = df_split["diagnosis"].values.astype(int)
    cnts    = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
    w_cls   = 1.0 / np.maximum(cnts, 1)
    s_wts   = torch.tensor([w_cls[l] for l in labels], dtype=torch.float)
    sampler = WeightedRandomSampler(s_wts, num_samples=len(s_wts), replacement=True)
    nw = min(4, os.cpu_count() or 1)
    return DataLoader(
        dataset, batch_size=batch_size, sampler=sampler,
        num_workers=nw, pin_memory=(DEVICE.type=="cuda"),
        drop_last=drop_last, persistent_workers=(nw>0))

def make_loader(dataset, batch_size, shuffle=False, drop_last=False):
    """Standard DataLoader for validation and test."""
    nw = min(4, os.cpu_count() or 1)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        num_workers=nw, pin_memory=(DEVICE.type=="cuda"),
        drop_last=drop_last, persistent_workers=(nw>0))

# Smoke test
_ds_test = DRDataset(df_trainval.head(32), get_val_transform(224), img_size=224, use_cache=True)
_ld_test = make_loader(_ds_test, batch_size=8)
_imgs, _labels = next(iter(_ld_test))
print(f"   Loader smoke test: imgs={list(_imgs.shape)}  labels={list(_labels.shape)}")
del _ds_test, _ld_test, _imgs, _labels
print("✅ Step 14 complete — DataLoaders ready.")


## 🏗️ Step 15 — Model Initialization (MANDATORY BASELINE)

```
Backbone : tf_efficientnetv2_b1 (ImageNet pretrained)
Head     : GlobalAvgPool → BatchNorm → Dense(256, ReLU) → Dropout(0.5) → Dense(5)
```


In [ ]:
class DRModel(nn.Module):
    """
    EfficientNetV2-B1 backbone with custom classification head.
    
    Architecture:
      backbone (pretrained EfficientNetV2-B1, global_pool='avg')
      → BatchNorm1d(features)
      → Linear(features, 256)
      → ReLU
      → Dropout(0.5)
      → Linear(256, 5)
    """
    def __init__(self, model_name=CFG["model_name"], num_classes=NUM_CLASSES,
                 pretrained=True, dropout=CFG["dropout"]):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained,
            num_classes=0, global_pool="avg")
        feat = self.backbone.num_features
        self.head = nn.Sequential(
            nn.BatchNorm1d(feat),
            nn.Linear(feat, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad_(False)

    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad_(True)

    def unfreeze_top(self, n=4):
        """Partial unfreeze: last n blocks + head layers."""
        self.freeze_backbone()
        if hasattr(self.backbone, "blocks"):
            for blk in list(self.backbone.blocks)[-n:]:
                for p in blk.parameters(): p.requires_grad_(True)
        for attr in ("conv_head","bn2","norm_head","norm"):
            if hasattr(self.backbone, attr):
                for p in getattr(self.backbone, attr).parameters():
                    p.requires_grad_(True)

# ── Sanity check ──────────────────────────────────────────────
_m  = DRModel(pretrained=False).to(DEVICE)
_x  = torch.randn(2, 3, 224, 224).to(DEVICE)
_o  = _m(_x)
assert _o.shape == (2, NUM_CLASSES), f"Wrong output shape: {_o.shape}"
total_p    = sum(p.numel() for p in _m.parameters())
trainable_p= sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"   Model output shape : {list(_o.shape)}  ✅")
print(f"   Total params       : {total_p/1e6:.2f}M")
print(f"   Trainable params   : {trainable_p/1e6:.2f}M")
del _m, _x, _o; gc.collect()
print("\n✅ Step 15 complete — Model architecture verified.")


## ⚖️ Step 16 — Loss + Optimizer + Scheduler

- **Loss**: 0.5 × Weighted CrossEntropy + 0.5 × Focal Loss (γ=2, α=0.25) + label_smoothing=0.05
- **Optimizer**: AdamW (weight_decay=1e-4)
- **Scheduler**: CosineAnnealingLR (3e-4 → 1e-5)


In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss for class imbalance. gamma=2, alpha=0.25."""
    def __init__(self, alpha=0.25, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.alpha=alpha; self.gamma=gamma
        self.weight=weight; self.ls=label_smoothing

    def forward(self, logits, targets):
        ce  = F.cross_entropy(logits, targets, weight=self.weight,
                              label_smoothing=self.ls, reduction="none")
        pt  = torch.exp(-ce)
        return (self.alpha * (1-pt)**self.gamma * ce).mean()

class HybridLoss(nn.Module):
    """
    0.5 × Weighted CrossEntropy (label_smoothing=0.05)
  + 0.5 × Focal Loss (gamma=2, alpha=0.25)
    Class weights applied in both terms.
    """
    def __init__(self, class_weights=None):
        super().__init__()
        w = class_weights
        self.ce    = nn.CrossEntropyLoss(weight=w, label_smoothing=CFG["label_smooth"])
        self.focal = FocalLoss(alpha=0.25, gamma=2.0, weight=w,
                               label_smoothing=CFG["label_smooth"])

    def forward(self, logits, targets):
        return 0.5*self.ce(logits,targets) + 0.5*self.focal(logits,targets)

# ── Metric ────────────────────────────────────────────────────
def qwk(y_true, y_pred):
    """Quadratic Weighted Kappa — primary metric."""
    return cohen_kappa_score(np.array(y_true), np.array(y_pred), weights="quadratic")

# ── Quick loss check ──────────────────────────────────────────
_criterion = HybridLoss(CLASS_WEIGHTS.to(DEVICE))
_logits    = torch.randn(4, NUM_CLASSES).to(DEVICE)
_targets   = torch.randint(0, NUM_CLASSES, (4,)).to(DEVICE)
_loss_val  = _criterion(_logits, _targets)
print(f"   HybridLoss smoke test: {_loss_val.item():.4f}  ✅")
del _criterion, _logits, _targets, _loss_val

print("\nOptimizer  : AdamW  lr=3e-4 → 1e-6, weight_decay=1e-4")
print("Scheduler  : CosineAnnealingLR per phase")
print("Loss       : 0.5×WCE + 0.5×Focal | label_smooth=0.05")
print("\n✅ Step 16 complete.")


## 💾 Step 17 — Checkpoint & Resume System (FULL RECOVERY)

Stores: epoch · batch · phase · model_state · optimizer_state · scheduler_state · scaler_state · best_qwk · thresholds · training_history


In [ ]:
def save_checkpoint(path, model, optimizer, scheduler, scaler,
                    epoch, batch, phase_id, best_qwk, thresholds, history):
    """
    Full checkpoint — supports exact batch-level resume.
    Stores everything needed to continue training from the exact point it stopped.
    """
    torch.save({
        "model"      : model.state_dict(),
        "optimizer"  : optimizer.state_dict(),
        "scheduler"  : scheduler.state_dict(),
        "scaler"     : scaler.state_dict() if scaler else None,
        "epoch"      : epoch,
        "batch"      : batch,
        "phase_id"   : phase_id,
        "best_qwk"   : best_qwk,
        "thresholds" : thresholds,
        "history"    : history,
    }, path)

def load_checkpoint(path, model, optimizer=None, scheduler=None,
                    scaler=None, map_location="cpu"):
    ckpt = safe_load(path, map_location)
    model.load_state_dict(ckpt["model"])
    if optimizer and ckpt.get("optimizer"): optimizer.load_state_dict(ckpt["optimizer"])
    if scheduler and ckpt.get("scheduler"): scheduler.load_state_dict(ckpt["scheduler"])
    if scaler    and ckpt.get("scaler"):    scaler.load_state_dict(ckpt["scaler"])
    return ckpt

print("Checkpoint stores:")
fields = ["model state_dict","optimizer state_dict","scheduler state_dict",
          "scaler state_dict","epoch","batch index","phase",
          "best_qwk","thresholds","training_history"]
for f in fields: print(f"  ✓ {f}")
print("\n✅ Step 17 complete.")


## 📊 Step 18 — Training State Management

In [ ]:
class TrainingState:
    """
    Tracks and persists all training state:
      - Per-epoch loss and QWK
      - Per-fold performance
      - Best model identification
      - Fold completion flags (resume-safe)
    """
    def __init__(self):
        self.fold_histories = {}   # fold → list of epoch dicts
        self.fold_best_qwks = {}   # fold → best QWK
        self.global_history  = []  # all epochs across all folds

    def record(self, fold, phase, epoch, tr_loss, val_qwk, val_acc):
        entry = dict(fold=fold, phase=phase, epoch=epoch,
                     tr_loss=tr_loss, val_qwk=val_qwk, val_acc=val_acc)
        self.fold_histories.setdefault(fold, []).append(entry)
        self.global_history.append(entry)

    def update_best(self, fold, qwk_val):
        prev = self.fold_best_qwks.get(fold, -1.0)
        if qwk_val > prev:
            self.fold_best_qwks[fold] = qwk_val
            return True
        return False

    def best_fold(self):
        if not self.fold_best_qwks: return 0
        return max(self.fold_best_qwks, key=self.fold_best_qwks.get)

    def summary(self):
        print("  Per-fold best QWK:")
        qwks = []
        for f in sorted(self.fold_best_qwks):
            q = self.fold_best_qwks[f]
            qwks.append(q)
            print(f"    Fold {f}: {q:.4f}")
        if qwks:
            print(f"  Mean: {np.mean(qwks):.4f} ± {np.std(qwks):.4f}")

    def plot_history(self):
        if not self.global_history: return
        df_hist = pd.DataFrame(self.global_history)
        fig, axes = plt.subplots(1,2, figsize=(14,4))
        for f in df_hist.fold.unique():
            fd = df_hist[df_hist.fold==f]
            axes[0].plot(fd.tr_loss.values, label=f"Fold {f}")
            axes[1].plot(fd.val_qwk.values, label=f"Fold {f}")
        axes[0].set_title("Training Loss",    fontweight="bold")
        axes[1].set_title("Validation QWK",   fontweight="bold")
        axes[0].set_xlabel("Epoch"); axes[1].set_xlabel("Epoch")
        for ax in axes: ax.legend(fontsize=7)
        plt.tight_layout()
        plt.savefig(str(PLOT_DIR/"training_history.png"), dpi=120, bbox_inches="tight")
        plt.show()

TRAIN_STATE = TrainingState()
print("✅ Step 18 complete — TrainingState initialized.")


## 🔁 Step 19 — Cross-Validation Training: 5-Fold Execution

Loops through all 5 folds, maintains class distribution, stores OOF predictions, tracks QWK per fold.


## 🏋️ Step 20 — Training: Phase-Wise Strategy

| Phase | Resolution | Epochs | Backbone | LR |
|-------|-----------|--------|----------|----|
| 1 | 224×224 | 15 | Frozen | 3e-4 |
| 2 | 384×384 | 40 | Partial (top-4 blocks) | 1e-4 |
| 3 | 512×512 | 25 | Full | 3.3e-5 |

> **Steps 19 + 20 + 21 + 22 are implemented in a single training cell below.**


## ✅ Step 21 — Validation (per epoch, per fold) | Step 22 — Early Stopping

In [ ]:
# ════════════════════════════════════════════════════════════════
#  STEPS 19 + 20 + 21 + 22 — Combined Training Cell
#  (5-Fold × 3-Phase × Early-Stopping × Validation)
# ════════════════════════════════════════════════════════════════

def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, device):
    """Step 20: one training epoch with AMP + grad clipping."""
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc="  train", leave=False)
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast("cuda"):
                loss = criterion(model(imgs), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            scaler.step(optimizer); scaler.update()
        else:
            loss = criterion(model(imgs), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    return total_loss / len(loader)

@torch.no_grad()
def validate_epoch(model, loader, scaler, device):
    """Step 21: validation — returns (softmax_probs, true_labels)."""
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in tqdm(loader, desc="  val  ", leave=False):
        imgs = imgs.to(device)
        if scaler:
            with torch.amp.autocast("cuda"): logits = model(imgs)
        else:
            logits = model(imgs)
        all_probs.append(F.softmax(logits, dim=1).cpu().float().numpy())
        all_labels.extend(labels.numpy())
    return np.concatenate(all_probs), np.array(all_labels)

# ── OOF storage ───────────────────────────────────────────────
OOF_PROBS  = np.zeros((len(df_trainval), NUM_CLASSES), dtype=np.float32)
OOF_LABELS = df_trainval["diagnosis"].values.copy()
scaler_global = torch.amp.GradScaler("cuda") if USE_AMP else None

print("=" * 70)
print(f"  5-FOLD CV | {CFG['model_name']} | device={DEVICE} | AMP={USE_AMP}")
print("=" * 70)

# ════════════════  FOLD LOOP (Steps 19 + 20 + 21 + 22)  ═══════
for fold in range(CFG["n_folds"]):

    ckpt_best = ARTIFACT_DIR / f"fold{fold}_best.pt"
    oof_file  = ARTIFACT_DIR / f"fold{fold}_oof.npy"
    done_flag = ARTIFACT_DIR / f"_done_fold{fold}.flag"

    # Resume: skip completed folds
    if done_flag.exists() and ckpt_best.exists():
        prev = safe_load(ckpt_best, "cpu")
        TRAIN_STATE.fold_best_qwks[fold] = prev.get("best_qwk", 0.0)
        if oof_file.exists():
            vi = df_trainval[df_trainval.fold==fold].index
            OOF_PROBS[vi] = np.load(str(oof_file))
        print(f"  ✅ [RESUME] Fold {fold} | QWK={TRAIN_STATE.fold_best_qwks[fold]:.4f}")
        continue

    print(f"\n  {'━'*25}  FOLD {fold}  {'━'*25}")
    df_tr  = df_trainval[df_trainval.fold!=fold].reset_index(drop=True)
    df_va  = df_trainval[df_trainval.fold==fold].reset_index(drop=True)
    val_idx = df_trainval[df_trainval.fold==fold].index

    model     = DRModel(pretrained=True).to(DEVICE)
    criterion = HybridLoss(CLASS_WEIGHTS.to(DEVICE))

    fold_best_qwk  = -1.0
    best_state     = None
    best_thresholds= [0.5, 1.5, 2.5, 3.5]   # default, updated per epoch
    fold_history   = []

    # ── Step 20: Phase-wise training loop ──────────────────────
    for phase in CFG["phases"]:
        pid, sz, bs, n_ep = phase["id"], phase["size"], phase["batch_size"], phase["epochs"]
        freeze = phase["freeze"]
        print(f"\n  Phase {pid} | {sz}px | {n_ep} epochs | bs={bs} | "
              f"{'backbone FROZEN' if freeze else 'backbone TRAINING'}")

        # Freeze / unfreeze backbone
        if freeze:
            model.freeze_backbone()
        elif pid == 2:
            model.unfreeze_top(n=4)       # partial: top-4 blocks
        else:
            model.unfreeze_backbone()     # full unfreeze in Phase 3

        tr_ds = DRDataset(df_tr, get_train_transform(sz), img_size=sz, use_cache=False)
        va_ds = DRDataset(df_va, get_val_transform(sz),   img_size=sz, use_cache=False)
        tr_ld = make_weighted_loader(df_tr, tr_ds, bs, drop_last=True)
        va_ld = make_loader(va_ds, bs)

        lr_phase  = CFG["lr"] / (3 ** (pid-1))   # 3e-4 → 1e-4 → 3.3e-5
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr_phase, weight_decay=CFG["weight_decay"])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=n_ep * len(tr_ld), eta_min=CFG["min_lr"])

        # ── Step 22: Early stopping per phase ──────────────────
        patience_cnt = 0

        for ep in range(n_ep):
            # ── Step 20: train ─────────────────────────────────
            tr_loss = train_one_epoch(
                model, tr_ld, criterion, optimizer, scheduler, scaler_global, DEVICE)

            # ── Step 21: validate ──────────────────────────────
            val_probs, val_labs = validate_epoch(model, va_ld, scaler_global, DEVICE)
            val_preds = val_probs.argmax(axis=1)
            val_qwk   = qwk(val_labs, val_preds)
            val_acc   = accuracy_score(val_labs, val_preds)

            # ── Step 18: record state ──────────────────────────
            TRAIN_STATE.record(fold, pid, ep, tr_loss, val_qwk, val_acc)
            fold_history.append(dict(phase=pid, epoch=ep, tr_loss=tr_loss,
                                     val_qwk=val_qwk, val_acc=val_acc))

            # Best model tracking
            improved = val_qwk > fold_best_qwk + CFG["min_delta"]
            if improved:
                fold_best_qwk = val_qwk
                best_state    = deepcopy(model.state_dict())
                patience_cnt  = 0
                save_checkpoint(
                    ckpt_best, model, optimizer, scheduler, scaler_global,
                    ep, 0, pid, fold_best_qwk, best_thresholds, fold_history)
                TRAIN_STATE.update_best(fold, fold_best_qwk)
            else:
                patience_cnt += 1

            star = " ★" if improved else ""
            print(f"    P{pid} Ep{ep+1:02d}/{n_ep}: "
                  f"loss={tr_loss:.4f}  QWK={val_qwk:.4f}  Acc={val_acc*100:.1f}%{star}")

            # ── Step 22: early stop check ─────────────────────
            if patience_cnt >= CFG["patience"]:
                print(f"    ↳ Early stop triggered (patience={CFG['patience']}, "
                      f"min_delta={CFG['min_delta']})")
                break

    print(f"  ✅ Fold {fold} done — Best QWK: {fold_best_qwk:.4f}")

    # ── Step 23 prep: collect OOF probs with best model ────────
    model.load_state_dict(best_state)
    va_ds_f = DRDataset(df_va, get_val_transform(512), img_size=512, use_cache=False)
    va_ld_f = make_loader(va_ds_f, batch_size=8)
    fold_probs, _ = validate_epoch(model, va_ld_f, scaler_global, DEVICE)
    OOF_PROBS[val_idx] = fold_probs[:len(val_idx)]
    np.save(str(oof_file), fold_probs[:len(val_idx)])

    done_flag.touch()
    del model; gc.collect()
    if DEVICE.type=="cuda": torch.cuda.empty_cache()

# Save full OOF
np.save(str(ARTIFACT_DIR/"oof_probs.npy"),  OOF_PROBS)
np.save(str(ARTIFACT_DIR/"oof_labels.npy"), OOF_LABELS)

print("\n" + "="*70)
TRAIN_STATE.summary()
print("="*70)
TRAIN_STATE.plot_history()
print("\n✅ Steps 19–22 complete — Training done.")


## 📦 Step 23 — OOF Predictions (Store + Verify)

In [ ]:
OOF_PROBS  = np.load(str(ARTIFACT_DIR/"oof_probs.npy"))
OOF_LABELS = np.load(str(ARTIFACT_DIR/"oof_labels.npy"))

print(f"OOF probs  shape : {OOF_PROBS.shape}")
print(f"OOF labels shape : {OOF_LABELS.shape}")
print(f"OOF label dist   : {dict(zip(*np.unique(OOF_LABELS, return_counts=True)))}")

# Baseline argmax QWK (no threshold optimization yet)
oof_preds_argmax = OOF_PROBS.argmax(axis=1)
oof_qwk_baseline = qwk(OOF_LABELS, oof_preds_argmax)
oof_acc_baseline = accuracy_score(OOF_LABELS, oof_preds_argmax)
print(f"\n  OOF QWK (argmax baseline) : {oof_qwk_baseline:.4f}")
print(f"  OOF Acc (argmax baseline) : {oof_acc_baseline*100:.2f}%")

st_save("oof_qwk_baseline", float(oof_qwk_baseline))
print("\n✅ Step 23 complete — OOF predictions verified.")


## 🔁 Step 24 — TTA (Test-Time Augmentation)

5-view TTA per model:
- Original
- Horizontal flip
- Vertical flip  
- Mild brightness variation
- ±10° rotation

All fold models averaged → ensemble.


In [ ]:
def get_tta_transforms(sz):
    base = [A.Resize(sz,sz), A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD), ToTensorV2()]
    return [
        A.Compose(base),                                                        # original
        A.Compose([A.HorizontalFlip(p=1.0)]   + base),                         # h-flip
        A.Compose([A.VerticalFlip(p=1.0)]     + base),                         # v-flip
        A.Compose([A.RandomBrightnessContrast(
            brightness_limit=0.1, contrast_limit=0.1, p=1.0)] + base),         # brightness
        A.Compose([A.Rotate(limit=10, p=1.0)] + base),                         # rotate
    ]

@torch.no_grad()
def tta_predict(df_infer, tta_size=512):
    """
    Ensemble TTA inference across all fold checkpoints.
    Returns averaged softmax probabilities of shape (N, NUM_CLASSES).
    """
    ckpt_paths = sorted(ARTIFACT_DIR.glob("fold*_best.pt"))
    if not ckpt_paths:
        raise RuntimeError("No fold checkpoints — run Steps 19–22 first.")

    tta_tfs  = get_tta_transforms(tta_size)
    ensemble = np.zeros((len(df_infer), NUM_CLASSES), dtype=np.float32)
    n_models = 0

    for ckpt_path in ckpt_paths:
        ckpt  = safe_load(ckpt_path, DEVICE)
        model = DRModel(pretrained=False).to(DEVICE)
        model.load_state_dict(ckpt["model"])
        model.eval()

        fold_sum = np.zeros((len(df_infer), NUM_CLASSES), dtype=np.float32)
        for tf in tta_tfs:
            ds = DRDataset(df_infer, tf, img_size=tta_size, use_cache=False)
            ld = make_loader(ds, batch_size=8)
            batch_probs = []
            for imgs, _ in tqdm(ld, desc=f"  {ckpt_path.stem} TTA", leave=False):
                imgs = imgs.to(DEVICE)
                if USE_AMP:
                    with torch.amp.autocast("cuda"): logits = model(imgs)
                else:
                    logits = model(imgs)
                batch_probs.append(F.softmax(logits,dim=1).cpu().float().numpy())
            fold_sum += np.concatenate(batch_probs)[:len(df_infer)]

        ensemble += fold_sum / len(tta_tfs)
        n_models += 1
        del model; gc.collect()
        if DEVICE.type=="cuda": torch.cuda.empty_cache()

    return ensemble / n_models

print(f"TTA pipeline ready: 5-view × {CFG['n_folds']} folds = "
      f"{5 * CFG['n_folds']} forward passes per test image")
print("✅ Step 24 complete — TTA function defined.")


## 🎯 Step 25 — Threshold Optimization (OOF → maximize QWK)

In [ ]:
class ThresholdOptimizer:
    """
    Optimizes 4 cut-point thresholds on the softmax scalar:
      scalar = probs @ [0,1,2,3,4]
    Then: grade = argbin(scalar, thresholds)
    Directly maximizes QWK via Nelder-Mead.
    """
    def __init__(self):
        self.thresholds_ = np.array([0.5, 1.5, 2.5, 3.5])

    def _predict_from_scalar(self, scalar, thresholds):
        cuts = np.sort(thresholds)
        return pd.cut(
            scalar,
            bins=[-np.inf]+list(cuts)+[np.inf],
            labels=list(range(NUM_CLASSES))
        ).astype(int).values

    def _loss(self, thresholds, scalar, y_true):
        preds = self._predict_from_scalar(scalar, thresholds)
        return -cohen_kappa_score(y_true, preds, weights="quadratic")

    def fit(self, probs, y_true):
        scalar = probs @ np.arange(NUM_CLASSES)
        res = minimize(
            self._loss, self.thresholds_, args=(scalar, y_true),
            method="Nelder-Mead",
            options={"maxiter":2000,"xatol":1e-6,"fatol":1e-9})
        self.thresholds_ = np.sort(res.x)
        return self

    def predict(self, probs):
        scalar = probs @ np.arange(NUM_CLASSES)
        return np.clip(self._predict_from_scalar(scalar, self.thresholds_), 0, 4)

# ── Fit on OOF ────────────────────────────────────────────────
print("Fitting thresholds on OOF predictions ...")
thr_opt = ThresholdOptimizer()
thr_opt.fit(OOF_PROBS, OOF_LABELS)

oof_preds_opt = thr_opt.predict(OOF_PROBS)
oof_qwk_opt   = qwk(OOF_LABELS, oof_preds_opt)
oof_acc_opt   = accuracy_score(OOF_LABELS, oof_preds_opt)

print(f"  OOF QWK (argmax)    : {oof_qwk_baseline:.4f}")
print(f"  OOF QWK (optimized) : {oof_qwk_opt:.4f}  ← IMPROVEMENT: "
      f"+{oof_qwk_opt-oof_qwk_baseline:.4f}")
print(f"  OOF Acc (optimized) : {oof_acc_opt*100:.2f}%")
print(f"  Optimal thresholds  : {np.round(thr_opt.thresholds_,3)}")

np.save(str(ARTIFACT_DIR/"oof_preds.npy"),   oof_preds_opt)
np.save(str(ARTIFACT_DIR/"thresholds.npy"),  thr_opt.thresholds_)
st_save("oof_qwk", float(oof_qwk_opt))
st_save("oof_acc", float(oof_acc_opt))

print(f"\n  Target QWK ≥ 0.90: {'✅ MET' if oof_qwk_opt>=0.90 else '⚠️  not yet'}")
print("\n✅ Step 25 complete — Threshold optimization done.")


## 🧪 Step 26 — Testing: Final Hold-Out Set (no data leakage)

In [ ]:
print(f"Running TTA inference on {len(df_test)} held-out test images ...")
print("(This set was NEVER used during training or threshold fitting)")
print()

TEST_PROBS = tta_predict(df_test, tta_size=512)

# Apply optimized thresholds
_thr = np.load(str(ARTIFACT_DIR/"thresholds.npy"))
thr_opt.thresholds_ = _thr
TEST_PREDS  = thr_opt.predict(TEST_PROBS)
TEST_LABELS = df_test["diagnosis"].values

np.save(str(ARTIFACT_DIR/"test_probs.npy"), TEST_PROBS)
np.save(str(ARTIFACT_DIR/"test_preds.npy"), TEST_PREDS)

test_qwk_val = qwk(TEST_LABELS, TEST_PREDS)
test_acc_val = accuracy_score(TEST_LABELS, TEST_PREDS)

print(f"  Test QWK : {test_qwk_val:.4f}  {'✅' if test_qwk_val>=0.90 else '⚠️ '}")
print(f"  Test Acc : {test_acc_val*100:.2f}%")

st_save("test_qwk", float(test_qwk_val))
st_save("test_acc", float(test_acc_val))
print("\n✅ Step 26 complete — Hold-out test evaluation done.")


## 📈 Step 27 — Metrics & Evaluation

- QWK (primary, target ≥ 0.90)
- Accuracy (target ≥ 85–90%)
- Confusion matrix
- Per-class recall & precision


In [ ]:
TEST_PROBS  = np.load(str(ARTIFACT_DIR/"test_probs.npy"))
TEST_PREDS  = np.load(str(ARTIFACT_DIR/"test_preds.npy"))
TEST_LABELS = df_test["diagnosis"].values

test_qwk_val = qwk(TEST_LABELS, TEST_PREDS)
test_acc_val = accuracy_score(TEST_LABELS, TEST_PREDS)

# ── Plot 1: Confusion Matrix + Per-class bar ──────────────────
cm  = confusion_matrix(TEST_LABELS, TEST_PREDS)
fig, axes = plt.subplots(1, 2, figsize=(16,6))

ConfusionMatrixDisplay(cm, display_labels=[f"G{i}" for i in range(5)]).plot(
    ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title(
    f"Test Confusion Matrix\nQWK={test_qwk_val:.4f}   Acc={test_acc_val*100:.1f}%",
    fontweight="bold")

report = classification_report(
    TEST_LABELS, TEST_PREDS,
    target_names=[f"G{i} {GRADE_MAP[i]}" for i in range(5)],
    output_dict=True)
recalls    = [report[f"G{i} {GRADE_MAP[i]}"]["recall"]    for i in range(5)]
precisions = [report[f"G{i} {GRADE_MAP[i]}"]["precision"] for i in range(5)]
x = np.arange(5); w = 0.35
axes[1].bar(x-w/2, recalls,    width=w, color=GRADE_COLORS, label="Recall",    alpha=0.9)
axes[1].bar(x+w/2, precisions, width=w, color=GRADE_COLORS, label="Precision", alpha=0.5, hatch="//")
axes[1].set_xticks(x); axes[1].set_xticklabels([f"G{i}" for i in range(5)])
axes[1].set_ylim(0, 1.15); axes[1].legend()
axes[1].set_title("Per-Class Recall & Precision", fontweight="bold")
plt.tight_layout()
plt.savefig(str(PLOT_DIR/"test_confusion_matrix.png"), dpi=120, bbox_inches="tight")
plt.show()

# ── Text report ───────────────────────────────────────────────
print(classification_report(
    TEST_LABELS, TEST_PREDS,
    target_names=[f"G{i} {GRADE_MAP[i]}" for i in range(5)]))

# ── QWK target check ──────────────────────────────────────────
print("-" * 50)
print(f"  QWK  : {test_qwk_val:.4f}  "
      f"{'✅ TARGET MET (≥0.90)' if test_qwk_val>=0.90 else '⚠️  below 0.90 — train more'}")
print(f"  Acc  : {test_acc_val*100:.2f}%  "
      f"{'✅' if test_acc_val>=0.85 else '⚠️  below 85%'}")
print("-" * 50)
print("\n✅ Step 27 complete.")


## 💾 Step 28 — Model Export (weights + thresholds + label mapping)

In [ ]:
import json as _json

# Best fold by QWK
_best_fold = TRAIN_STATE.best_fold()
_src_ckpt  = ARTIFACT_DIR / f"fold{_best_fold}_best.pt"

# Load best model weights
_ckpt = safe_load(_src_ckpt, "cpu")
_exp_model = DRModel(pretrained=False)
_exp_model.load_state_dict(_ckpt["model"])
_exp_model.eval()

# ── 1. Full export bundle (model + meta) ───────────────────────
_bundle_path = EXPORT_DIR / "dr_model_final.pt"
torch.save({
    "model_state_dict" : _exp_model.state_dict(),
    "model_name"       : CFG["model_name"],
    "num_classes"      : NUM_CLASSES,
    "grade_map"        : GRADE_MAP,
    "thresholds"       : thr_opt.thresholds_.tolist(),
    "imagenet_mean"    : IMAGENET_MEAN,
    "imagenet_std"     : IMAGENET_STD,
    "test_qwk"         : float(test_qwk_val),
    "oof_qwk"          : float(oof_qwk_opt),
    "best_fold"        : _best_fold,
}, str(_bundle_path))

# ── 2. Thresholds JSON ────────────────────────────────────────
_thr_path = EXPORT_DIR / "thresholds.json"
_thr_path.write_text(_json.dumps({
    "thresholds" : thr_opt.thresholds_.tolist(),
    "method"     : "Nelder-Mead OOF optimization",
    "oof_qwk"    : float(oof_qwk_opt),
}, indent=2))

# ── 3. Label mapping JSON ─────────────────────────────────────
_label_path = EXPORT_DIR / "label_map.json"
_label_path.write_text(_json.dumps(
    {str(k): v for k,v in GRADE_MAP.items()}, indent=2))

# ── 4. Config JSON ────────────────────────────────────────────
_cfg_path = EXPORT_DIR / "config.json"
_cfg_path.write_text(_json.dumps(CFG, indent=2))

print("Exported files:")
for f in sorted(EXPORT_DIR.glob("*")):
    sz = f.stat().st_size
    print(f"  ✅ {f.name:<30s}  {sz/1e6:.2f} MB" if sz>1e4 else f"  ✅ {f.name}")

del _exp_model; gc.collect()
print(f"\n  Best fold : {_best_fold}  "
      f"(QWK={TRAIN_STATE.fold_best_qwks.get(_best_fold,0):.4f})")
print("\n✅ Step 28 complete — Model exported.")


## 🎨 Step 29 — Grad-CAM++ Explainability

Visualizes model attention on lesion areas to validate clinical relevance.


In [ ]:
try:
    from pytorch_grad_cam import GradCAMPlusPlus
    from pytorch_grad_cam.utils.image import show_cam_on_image

    _bfi  = TRAIN_STATE.best_fold()
    _ckpt = safe_load(ARTIFACT_DIR/f"fold{_bfi}_best.pt", DEVICE)
    _m    = DRModel(pretrained=False).to(DEVICE)
    _m.load_state_dict(_ckpt["model"])
    _m.eval()

    # Robust target layer detection (EfficientNet / other backbones)
    if hasattr(_m.backbone, "blocks"):
        _tgt_layers = [_m.backbone.blocks[-1][-1]]
    else:
        _kids = list(_m.backbone.children())
        _tgt_layers = [_kids[-1] if isinstance(_kids[-1], nn.Module) else _kids[-2]]

    cam = GradCAMPlusPlus(model=_m, target_layers=_tgt_layers)

    fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(22, 9))
    for grade in range(NUM_CLASSES):
        sample = df[df.diagnosis==grade].sample(1, random_state=42).iloc[0]
        raw    = preprocess_fundus(sample.path, size=512)
        tf     = get_val_transform(512)
        tensor = tf(image=raw)["image"].unsqueeze(0).to(DEVICE)

        gc_map = cam(input_tensor=tensor, targets=None)[0]
        vis    = show_cam_on_image(raw.astype(np.float32)/255.0, gc_map, use_rgb=True)

        axes[0][grade].imshow(raw)
        axes[0][grade].set_title(f"G{grade}: {GRADE_MAP[grade]}", fontsize=9, fontweight="bold")
        axes[0][grade].axis("off")
        axes[1][grade].imshow(vis)
        axes[1][grade].set_title("Grad-CAM++ Attention", fontsize=8)
        axes[1][grade].axis("off")

    plt.suptitle("Grad-CAM++ — Model Lesion Attention per DR Grade",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(str(PLOT_DIR/"gradcam_all_grades.png"), dpi=120, bbox_inches="tight")
    plt.show()
    del _m, cam; gc.collect()
    print("✅ Step 29 complete — Grad-CAM++ saved.")

except ImportError:
    print("⚠️  pytorch-grad-cam not installed.")
    print("   Install: pip install pytorch-grad-cam")
    print("   Step 29 skipped gracefully — all other steps unaffected.")
except Exception as e:
    print(f"⚠️  Grad-CAM error: {e}")


## 🚀 Step 30 — Deployment: Streamlit App + Hugging Face

Generates `dr_app.py` — run with `streamlit run dr_app.py`


In [ ]:
APP_CODE = '''
import streamlit as st
import torch, cv2, numpy as np, json, pandas as pd
from pathlib import Path
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ── Paths ─────────────────────────────────────────────────────
EXPORT_DIR    = Path(__file__).parent / "export"
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
GRADE_MAP     = {0:"No DR",1:"Mild",2:"Moderate",3:"Severe",4:"Proliferative"}
GRADE_COLORS  = ["#2ecc71","#f1c40f","#e67e22","#e74c3c","#8e44ad"]

# ── Load model ────────────────────────────────────────────────
@st.cache_resource
def load_model():
    bundle = torch.load(
        str(EXPORT_DIR/"dr_model_final.pt"),
        map_location=DEVICE, weights_only=False)

    class DRModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.backbone = timm.create_model(
                bundle["model_name"], pretrained=False,
                num_classes=0, global_pool="avg")
            feat = self.backbone.num_features
            self.head = nn.Sequential(
                nn.BatchNorm1d(feat), nn.Linear(feat,256),
                nn.ReLU(True), nn.Dropout(0.5), nn.Linear(256,5))
        def forward(self, x): return self.head(self.backbone(x))

    m = DRModel().to(DEVICE)
    m.load_state_dict(bundle["model_state_dict"])
    m.eval()
    return m, np.array(bundle["thresholds"])

# ── Preprocessing ─────────────────────────────────────────────
def preprocess(img_rgb, size=512):
    img = cv2.resize(img_rgb, (size,size))
    cmask = np.zeros((size,size),np.uint8)
    cv2.circle(cmask,(size//2,size//2),int(size//2*0.97),255,-1)
    img[cmask==0] = 0
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    lab[:,:,0] = cv2.createCLAHE(2.0,(8,8)).apply(lab[:,:,0])
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    blur = cv2.GaussianBlur(img,(0,0),sigmaX=51)
    img  = cv2.addWeighted(img,4,blur,-4,128)
    img[cmask==0] = 0
    return img

# ── Inference ─────────────────────────────────────────────────
def predict(model, thresholds, img_rgb):
    proc   = preprocess(img_rgb)
    tf     = A.Compose([A.Resize(512,512),
                        A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),
                        ToTensorV2()])
    tensor = tf(image=proc)["image"].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = F.softmax(model(tensor),dim=1).cpu().numpy()[0]
    scalar = probs @ np.arange(5)
    grade  = int(pd.cut([scalar],
                  bins=[-np.inf]+list(np.sort(thresholds))+[np.inf],
                  labels=[0,1,2,3,4]).astype(int)[0])
    return grade, probs

# ── UI ────────────────────────────────────────────────────────
st.set_page_config(page_title="DR Grading", page_icon="🩺", layout="wide")
st.title("🩺 Diabetic Retinopathy Grading")
st.caption("Upload a fundus photograph — AI returns DR grade 0-4 with confidence scores")

col1, col2 = st.columns([1,1])
with col1:
    uploaded = st.file_uploader("Upload fundus image", type=["png","jpg","jpeg"])

if uploaded:
    model, thresholds = load_model()
    img_rgb = np.array(Image.open(uploaded).convert("RGB"))
    with col1:
        st.image(img_rgb, caption="Input", use_container_width=True)
    with st.spinner("Analyzing ..."):
        grade, probs = predict(model, thresholds, img_rgb)
    with col2:
        st.subheader(f"Grade {grade}: {GRADE_MAP[grade]}")
        conf = float(probs[grade])
        st.progress(int(conf*100), text=f"Confidence: {conf*100:.1f}%")
        st.subheader("Confidence per Grade")
        chart_df = pd.DataFrame({
            "Grade":[f"G{i}: {GRADE_MAP[i]}" for i in range(5)],
            "Probability": probs})
        st.bar_chart(chart_df.set_index("Grade"))
        if grade == 0:
            st.success("✅ No Diabetic Retinopathy detected.")
        elif grade == 1:
            st.info("ℹ️  Mild DR — recommend follow-up in 12 months.")
        elif grade == 2:
            st.warning("⚠️  Moderate DR — refer to ophthalmologist.")
        elif grade == 3:
            st.error("🚨 Severe DR — urgent ophthalmology referral.")
        else:
            st.error("🚨 Proliferative DR — immediate treatment required.")

st.markdown("---")
st.caption("⚠️  For research use only. Not for clinical diagnosis.")
'''

# Save app file
_app_path = EXPORT_DIR / "dr_app.py"
_app_path.write_text(APP_CODE.strip())

# Requirements file for Hugging Face / local
_req_path = EXPORT_DIR / "requirements.txt"
_req_path.write_text(
    "torch\ntimm\nalbumentations\nopencv-python-headless\nstreamlit\nnumpy\npandas\n")

# README for Hugging Face Spaces
_readme = EXPORT_DIR / "README.md"
_readme.write_text("""---
title: Diabetic Retinopathy Grading
emoji: 🩺
colorFrom: green
colorTo: red
sdk: streamlit
sdk_version: "1.35.0"
app_file: dr_app.py
pinned: false
---

# 🩺 Diabetic Retinopathy Grading

Upload a fundus image → get DR grade 0-4 with confidence scores.

**Model**: EfficientNetV2-B1 | **Dataset**: APTOS 2019  
**Metric**: Quadratic Weighted Kappa (QWK)

> ⚠️ Research use only. Not for clinical diagnosis.
""")

print("Deployment files created:")
for f in sorted(EXPORT_DIR.glob("*")):
    print(f"  ✅ {f.name}")

print()
print("To run locally:")
print(f"  streamlit run {_app_path}")
print()
print("To deploy on Hugging Face Spaces:")
print("  1. Go to https://huggingface.co/new-space")
print("  2. SDK: Streamlit")
print(f"  3. Upload all files from: {EXPORT_DIR}")
print("  4. The Space will auto-install requirements.txt and launch dr_app.py")
print("\n✅ Step 30 complete — Deployment ready.")


## 📋 Final Summary — All 30 Steps

In [ ]:
state = st_load()
W = 70
print("=" * W)
print("  DIABETIC RETINOPATHY GRADING — v19 PRODUCTION — ALL 30 STEPS")
print("=" * W)
print(f"  Backbone   : {CFG['model_name']}")
print(f"  Loss       : HybridLoss (0.5×WCE + 0.5×Focal) + label_smooth={CFG['label_smooth']}")
print(f"  Device     : {DEVICE} | AMP: {'ON' if USE_AMP else 'OFF'}")
print(f"  Dataset    : APTOS 2019")
print(f"  Clean imgs : {len(df):,}")
print(f"  Train+Val  : {len(df_trainval):,}")
print(f"  Test       : {len(df_test):,}  (held-out)")
print()
print("  ─── K-Fold Cross-Validation Results ───")
TRAIN_STATE.summary()
print()
print(f"  OOF QWK (argmax)    : {state.get('oof_qwk_baseline','N/A')}")
print(f"  OOF QWK (optimized) : {state.get('oof_qwk','N/A')}")
print(f"  OOF Acc             : {float(state.get('oof_acc',0))*100:.2f}%")
print()
print("  ─── Final Hold-Out Test ───")
tqwk = float(state.get('test_qwk',0))
tacc = float(state.get('test_acc',0))
print(f"  Test QWK : {tqwk:.4f}  {'✅ TARGET MET' if tqwk>=0.90 else '⚠️  below 0.90'}")
print(f"  Test Acc : {tacc*100:.2f}%  {'✅' if tacc>=0.85 else '⚠️  below 85%'}")
print()
print("  ─── 30-Step Checklist ───")
steps = [
    "Setup (GPU/CPU/Storage)",
    "Install Requirements",
    "Kaggle Auth",
    "Dataset Download & Extraction",
    "Load Dataset",
    "Data Cleaning (Laplacian+intensity)",
    "EDA",
    "Label Analysis",
    "Preprocessing (strict pipeline)",
    "Preprocessing Cache",
    "Train/Test Split",
    "K-Fold (StratifiedKFold=5)",
    "Augmentation + Dataset Pipeline",
    "DataLoader Creation",
    "Model Initialization",
    "Loss + Optimizer + Scheduler",
    "Checkpoint & Resume System",
    "Training State Management",
    "Cross-Validation Training",
    "Training (Phase-Wise)",
    "Validation (per epoch/fold)",
    "Early Stopping",
    "OOF Predictions",
    "TTA (5-view ensemble)",
    "Threshold Optimization (Nelder-Mead)",
    "Testing (Hold-Out)",
    "Metrics & Evaluation",
    "Model Export",
    "Grad-CAM++ Explainability",
    "Deployment (Streamlit + HuggingFace)",
]
for i, s in enumerate(steps, 1):
    print(f"  ✅ Step {i:02d}: {s}")
print()
print("=" * W)
print("  ⚠️  RESEARCH USE ONLY — NOT FOR CLINICAL DEPLOYMENT")
print("=" * W)
